# RePaint-part Pipeline — PoinTr + PointNet part-seg + part-conditioned RePaint

**System B** of a two-system comparison. This is *your existing pipeline*, ported onto the shared
evaluation contract so its numbers can be placed beside System A's
(`Joint6D_Completion_kaggle.ipynb`) without any reconciliation step.

```
partial (xyz only)  ──►  frozen PoinTr  ──►  completed geometry
                                                    │
partial (xyz+rgb) ──────────────────────────────────┤
                                                    ▼
                          PointNet part-seg  ──►  part-conditioned RePaint  ──►  colour
```

Six colouring methods are scored on **identical geometry**, which is the design's real strength —
it isolates the colouring question from the completion question:

| method | what it is |
|---|---|
| `NN-copy` | each filled point takes the colour of the nearest visible point |
| `part-mean` | per-predicted-part mean of the visible colours |
| `RePaint-vanilla` | colour DDPM, part-blind (`part_cond=False`) |
| `RePaint-part` | colour DDPM with part pooling + part one-hot, PointNet labels |
| `RePaint-part (oracle seg)` | the same model given **perfect** part labels |
| `oracle ceiling` | per-part **ground-truth** mean colour — the best any part-constant rule can do |

## Read this before comparing the two systems

**PoinTr's checkpoint was trained on ShapeNet-55, which contains these object categories — plausibly
these exact models.** System B's geometry stage may therefore have memorised part of the evaluation
set, while System A trains from scratch on 150 shapes. This biases the geometry comparison **in
favour of System B**, and no amount of careful metric design fixes it. Treat System B's geometry
numbers as an optimistic bound, and weight the *colour* comparison — where both systems generate
from scratch — much more heavily. The comparison report repeats this warning next to the table.

Two further asymmetries the report also carries:

* **Different point counts.** System B returns visible + PoinTr's fixed-size completion; System A
  returns the cloud it diffused. `EQUALIZE_PRED_N` in the shared spec puts both at the GT's own
  count before scoring, and `n_pred` is recorded either way so the choice stays visible.
* **The colour DDPM never sees occlusion during training** — it is trained unconditionally on
  complete clouds and only meets the mask at inference, via RePaint. System A trains with the mask
  in the loop. That is a genuine design difference between the systems, not a bug in either.

## §1 · Configuration

In [ ]:
# ======================= SHARED — must match System A =========================
SYSTEM_NAME   = "repaint_part"
CATEGORY      = "airplane"
DIFFICULTIES  = ["moderate"]
NUM_POINTS    = 2048
NUM_TRAIN     = 150
NUM_VAL       = 40
SEED          = 0
FSCORE_TAU    = 0.01
MATCH_TAU     = 0.02
EQUALIZE_PRED_N = "gt"
EVAL_REPEATS  = 3
# ==============================================================================

# ------------------- SYSTEM B ONLY --------------------------------------------
GEOMETRY_SOURCE = "pointr"
# "pointr"     the real pipeline: frozen PoinTr completes the geometry
# "gt_oracle"  PERFECT geometry (the GT missing points). Isolates the COLOURING method from the
#              completion method and gives every colour metric an upper bound.

ALLOW_GEOMETRY_FALLBACK = False
# If PoinTr fails to load with GEOMETRY_SOURCE="pointr", STOP rather than silently substituting
# perfect geometry. A fallback run produces cd_l1=0 / fscore=1.0 rows that look like results and
# are not — and it is easy to miss one warning line in a 4000-line log. Set True only when you
# deliberately want the colour-only comparison.

METHODS = ["NN-copy", "part-mean", "RePaint-vanilla", "RePaint-part",
           "RePaint-part (oracle seg)", "oracle ceiling"]

POINTR_REPO   = "https://github.com/eylulpelinkilic/Pelin_Efe_PoinTr.git"
POINTR_CKPT_ID = "1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ"     # PoinTr_ShapeNet55.pth (gdown)
POINTR_IN     = 2048       # points fed to PoinTr
COMP_POINTS   = 2048       # PoinTr's 6144 output is FPS-reduced to this
FRAME_PROBE   = 3          # objects used by the 48-candidate axis search

SEG_EPOCHS    = 60
SEG_LR        = 1e-3
NUM_PARTS     = 4

DDPM_T        = 200
DDPM_EPOCHS   = 300
DDPM_WIDTH    = 128
DDPM_K        = 16
DDPM_BLOCKS   = 3
MATCH_DDPM_PARAMS = True   # widen the part-BLIND model so P2 compares architecture,
                           # not capacity (native gap is +11.2%)
DDPM_LR       = 2e-4
DDPM_BS       = 8
JUMP_LENGTH   = 10
JUMP_N_SAMPLE = 3
ADAPTIVE_JUMPS = True

N_BOOT        = 2000
RUN_FULL_EXPERIMENT = False
PILOT = False
# PILOT=True runs everything end to end in minutes — PoinTr setup is the one stage it cannot fake,
# so run it once with GEOMETRY_SOURCE="gt_oracle" to validate scoring/figures/report, then again
# with "pointr" to validate the geometry stage.
# ==============================================================================

if PILOT:
    NUM_TRAIN, NUM_VAL, EVAL_REPEATS = 24, 8, 1
    SEG_EPOCHS, DDPM_EPOCHS, DDPM_T = 4, 20, 40
    RUN_FULL_EXPERIMENT = True     # forced: a pilot that stops early tests nothing
    print(">>> PILOT MODE — results are meaningless; this only proves the plumbing works.")
    print(">>> RUN_FULL_EXPERIMENT forced True so inference, metrics and every figure execute.\n")

import os, sys, math, json, time, glob, random, warnings, hashlib, subprocess, shutil
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)

ON_KAGGLE   = os.path.isdir("/kaggle")
RESULTS_DIR = Path("/kaggle/working/results_pipeline" if ON_KAGGLE else "./results_pipeline")
WORK = Path("/kaggle/temp/pcc_work" if ON_KAGGLE else "./_work")
for sub in ("figures", "tables", "checkpoints"):
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

SYNSET = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}[CATEGORY]
HF_REPO = "eylulpelinkilic/Colored_Point_Clouds"

print(f"SYSTEM      : {SYSTEM_NAME}")
print(f"geometry    : {GEOMETRY_SOURCE}" + ("" if GEOMETRY_SOURCE != "pointr" else
      f"   (fallback {'ALLOWED' if ALLOW_GEOMETRY_FALLBACK else 'BLOCKED'})"))
print(f"category    : {CATEGORY} ({SYNSET}) | difficulties {DIFFICULTIES}")
print(f"points      : {NUM_POINTS} | {NUM_TRAIN} train / {NUM_VAL} val | seed {SEED}")
print(f"methods     : {len(METHODS)} — {METHODS}")
print(f"RUN_FULL_EXPERIMENT = {RUN_FULL_EXPERIMENT}")

## §2 · Environment

In [ ]:
import importlib
for m in ["numpy", "torch", "pandas", "scipy", "matplotlib", "huggingface_hub", "plotly"]:
    print(f"{m:16s} {'present' if importlib.util.find_spec(m) else 'MISSING'}")
print()
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy import stats

print(f"numpy {np.__version__} | torch {torch.__version__} | pandas {pd.__version__}")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cuda: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
else:
    print("  (CPU — PoinTr will be very slow; use GEOMETRY_SOURCE='gt_oracle' to test)")

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
print(f"\nseeded with {SEED} | device {DEV}")

## Shared spec — the control that makes this comparison mean anything

Two systems can only be compared if they are scored on the same objects, in the same frame, by the
same code. That is not achievable by "using the same settings" — it drifts the moment one notebook
is edited. So everything shared is defined **once**, embedded verbatim as a string, and hashed:

* data loading and the partial/`_missing` disambiguation
* the deterministic train/val split
* the **partial-frame normalisation** (never ground-truth statistics — see below)
* all 11 metrics, and `spec_score`, the single scoring entry point
* `EQUALIZE_PRED_N`, the density control

Each notebook prints `SPEC_HASH`. **If two notebooks print different hashes, their numbers must not
be compared** — and the comparison notebook refuses to merge them. `spec_config_fingerprint()` does
the same for the settings that must agree (category, difficulty, point count, split sizes, seed,
thresholds).

The spec is also written to `results/shared_spec.py` so it can be read and diffed directly.

**Why the frame comes from the partial alone.** At test time only the partial exists. Normalising by
ground-truth centroid and radius would hand every system the object's true centre and extent — most
of the completion task — and would flatter whichever system exploits it more. One consequence is
visible downstream: some GT points then lie beyond radius 1, so geometry is never clamped.

In [ ]:
SPEC_SRC = r'''
# ============================================================================
# SHARED SPEC v3 — byte-identical in every notebook of this comparison.
# Data loading, the train/val split, the partial-frame normalisation, and every
# metric live HERE and nowhere else. Each notebook prints SPEC_HASH; if two
# notebooks print different hashes their numbers MUST NOT be compared.
# ============================================================================
SPEC_VERSION = 3

_PLY_NP = {"char": "i1", "uchar": "u1", "short": "i2", "ushort": "u2",
           "int": "i4", "uint": "u4", "float": "f4", "double": "f8",
           "int8": "i1", "uint8": "u1", "int16": "i2", "uint16": "u2",
           "int32": "i4", "uint32": "u4", "float32": "f4", "float64": "f8"}


def read_ply_xyzrgb(path):
    """Binary/ascii PLY -> (xyz float32 (N,3), rgb float32 (N,3) in [0,1]).

    The header is parsed rather than assumed, so an unexpected layout raises instead of
    silently producing garbage. Avoids open3d, which is broken in some local envs.
    """
    with open(path, "rb") as fh:
        if fh.readline().strip() != b"ply":
            raise ValueError(f"{path}: not a PLY")
        fmt, props, n, in_vertex = None, [], None, False
        while True:
            ln = fh.readline()
            if not ln:
                raise ValueError(f"{path}: header never ended")
            t = ln.strip().split()
            if not t:
                continue
            k = t[0].lower()
            if k == b"format":
                fmt = t[1].decode()
            elif k == b"element":
                in_vertex = (t[1].lower() == b"vertex")
                if in_vertex:
                    n = int(t[2])
            elif k == b"property" and in_vertex:
                if t[1].lower() == b"list":
                    raise ValueError(f"{path}: list property in vertex element")
                props.append((t[2].decode(), _PLY_NP[t[1].decode().lower()]))
            elif k == b"end_header":
                break
        names = [p[0] for p in props]
        for need in ("x", "y", "z"):
            if need not in names:
                raise ValueError(f"{path}: no '{need}' property (have {names})")
        if fmt == "binary_little_endian":
            arr = np.frombuffer(fh.read(), dtype=np.dtype([(a, "<" + b) for a, b in props]), count=n)
        elif fmt == "binary_big_endian":
            arr = np.frombuffer(fh.read(), dtype=np.dtype([(a, ">" + b) for a, b in props]), count=n)
        elif fmt == "ascii":
            arr = np.loadtxt(fh, dtype=np.dtype([(a, b) for a, b in props]), max_rows=n)
        else:
            raise ValueError(f"{path}: unknown format {fmt}")
    xyz = np.stack([arr["x"], arr["y"], arr["z"]], -1).astype(np.float32)
    cn = [c for c in ("red", "green", "blue") if c in names]
    if len(cn) == 3:
        rgb = np.stack([arr[c] for c in cn], -1).astype(np.float32)
        if any(dict(props)[c] == "u1" for c in cn):
            rgb /= 255.0
    else:
        rgb = np.full_like(xyz, 0.5)
    return xyz, np.clip(rgb, 0, 1)


def spec_fetch_data():
    """Local tree or HuggingFace snapshot holding labeled_s3/ and occluded_occ/."""
    for cand in [Path("data"), Path("/kaggle/input/colored-point-clouds"),
                 Path(os.environ.get("PCC_DATA_ROOT", "___none___"))]:
        if (cand / "labeled_s3" / SYNSET).is_dir() and (cand / "occluded_occ").is_dir():
            return cand, "local"
    from huggingface_hub import snapshot_download
    pats = [f"labeled_s3/{SYNSET}/*.npz"] + [f"occluded_occ/{d}/{SYNSET}/*.ply" for d in DIFFICULTIES]
    return Path(snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=pats)), "hf"


def spec_model_ids(root):
    """Model ids, sorted, with the partial/_missing trap handled and asserted."""
    all_ply = sorted((root / "occluded_occ" / DIFFICULTIES[0] / SYNSET).glob("*.ply"))
    partial = [p for p in all_ply if not p.stem.endswith("_missing")]
    assert len(partial) * 2 == len(all_ply), (
        f"partial/_missing pairing is not 1:1 ({len(partial)} vs {len(all_ply)}) — "
        "a naive glob('*.ply') returns BOTH files per model")
    return [p.stem for p in partial], len(all_ply)


def spec_visible_mask(gt_xyz, part_xyz, tol=1e-5):
    """Which GT rows appear in the partial. The partial is an exact subset (verified), so
    this mask is exact — it is not a fuzzy nearest-neighbour assignment."""
    d, i = cKDTree(gt_xyz).query(part_xyz, k=1)
    ok = d < tol
    m = np.zeros(len(gt_xyz), bool)
    m[i[ok]] = True
    return m, dict(matched=int(ok.sum()), n_partial=len(part_xyz),
                   max_d=float(d.max()), unique=int(len(np.unique(i[ok]))))


def spec_load_one(root, mid, difficulty):
    z = np.load(root / "labeled_s3" / SYNSET / f"{mid}.npz")
    xyz, rgb = z["xyz"].astype(np.float32), z["rgb"].astype(np.float32)
    part = z["part"].astype(np.int64)
    pxyz, _ = read_ply_xyzrgb(root / "occluded_occ" / difficulty / SYNSET / f"{mid}.ply")
    m, diag = spec_visible_mask(xyz, pxyz)
    return dict(mid=mid, xyz=xyz, rgb=rgb, part=part, vis=m, diag=diag)


def spec_subsample(n_have, n_want, rng):
    if n_have >= n_want:
        return rng.choice(n_have, n_want, replace=False)
    return np.concatenate([np.arange(n_have), rng.choice(n_have, n_want - n_have, replace=True)])


def spec_build_entry(root, mid, difficulty, rng):
    """One completion example in the PARTIAL's frame.

    The frame comes from the partial ALONE. Normalising by ground-truth centroid/radius
    would hand any model the object's true centre and extent — most of the completion task.
    Consequence: GT points can lie beyond radius 1, so geometry is never clamped downstream.
    """
    r = spec_load_one(root, mid, difficulty)
    idx = spec_subsample(len(r["xyz"]), NUM_POINTS, rng)
    xyz, rgb, part, vis = r["xyz"][idx], r["rgb"][idx], r["part"][idx], r["vis"][idx]
    if vis.sum() < 16 or (~vis).sum() < 16:
        return None
    mu = xyz[vis].mean(0)
    rad = max(float(np.linalg.norm(xyz[vis] - mu, axis=1).max()), 1e-6)
    g = (xyz - mu) / rad
    c = 2.0 * rgb - 1.0
    return dict(mid=mid, difficulty=difficulty,
                x0=np.concatenate([g, c], -1).astype(np.float32),
                vis=vis, part=part, mu=mu.astype(np.float32), rad=np.float32(rad),
                sub_idx=idx)


def spec_split(root):
    """The train/val split. Deterministic in SEED and identical across notebooks."""
    ids, n_ply = spec_model_ids(root)
    need = NUM_TRAIN + NUM_VAL
    if len(ids) < need:
        raise RuntimeError(f"only {len(ids)} models available, need {need}")
    ids = ids[:need]
    out = {}
    for tag, sel in (("train", ids[:NUM_TRAIN]), ("val", ids[NUM_TRAIN:])):
        rng = np.random.default_rng(abs(hash((SEED, tag))) % (2 ** 32))
        rows = []
        for mid in sel:
            for d in DIFFICULTIES:
                e = spec_build_entry(root, mid, d, rng)
                if e is not None:
                    rows.append(e)
        out[tag] = rows
    assert not (set(e["mid"] for e in out["train"]) & set(e["mid"] for e in out["val"])), \
        "train/val model overlap"
    return out, ids, n_ply


# ---------------------------------------------------------------- colour science
def srgb_to_lab(rgb):
    rgb = np.clip(np.asarray(rgb, np.float64), 0, 1)
    lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    xyz = lin @ M.T / np.array([0.95047, 1.0, 1.08883])
    e, k = 216 / 24389, 24389 / 27
    f = np.where(xyz > e, np.cbrt(xyz), (k * xyz + 16) / 116)
    return np.stack([116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]),
                     200 * (f[..., 1] - f[..., 2])], -1)


def ciede2000(lab1, lab2):
    L1, a1, b1 = lab1[..., 0], lab1[..., 1], lab1[..., 2]
    L2, a2, b2 = lab2[..., 0], lab2[..., 1], lab2[..., 2]
    C1, C2 = np.hypot(a1, b1), np.hypot(a2, b2)
    Cb = (C1 + C2) / 2
    G = 0.5 * (1 - np.sqrt(Cb ** 7 / (Cb ** 7 + 25.0 ** 7 + 1e-30)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p, C2p = np.hypot(a1p, b1), np.hypot(a2p, b2)
    h1p, h2p = np.degrees(np.arctan2(b1, a1p)) % 360, np.degrees(np.arctan2(b2, a2p)) % 360
    dLp, dCp = L2 - L1, C2p - C1p
    dh = h2p - h1p
    dh = np.where(C1p * C2p == 0, 0, np.where(dh > 180, dh - 360, np.where(dh < -180, dh + 360, dh)))
    dHp = 2 * np.sqrt(C1p * C2p) * np.sin(np.radians(dh / 2))
    Lbp, Cbp = (L1 + L2) / 2, (C1p + C2p) / 2
    hs = h1p + h2p
    hbp = np.where(C1p * C2p == 0, hs,
                   np.where(np.abs(h1p - h2p) <= 180, hs / 2,
                            np.where(hs < 360, (hs + 360) / 2, (hs - 360) / 2)))
    T = (1 - 0.17 * np.cos(np.radians(hbp - 30)) + 0.24 * np.cos(np.radians(2 * hbp))
         + 0.32 * np.cos(np.radians(3 * hbp + 6)) - 0.20 * np.cos(np.radians(4 * hbp - 63)))
    dth = 30 * np.exp(-(((hbp - 275) / 25) ** 2))
    Rc = 2 * np.sqrt(Cbp ** 7 / (Cbp ** 7 + 25.0 ** 7 + 1e-30))
    Sl = 1 + 0.015 * (Lbp - 50) ** 2 / np.sqrt(20 + (Lbp - 50) ** 2)
    Sc, Sh = 1 + 0.045 * Cbp, 1 + 0.015 * Cbp * T
    Rt = -np.sin(np.radians(2 * dth)) * Rc
    return np.sqrt((dLp / Sl) ** 2 + (dCp / Sc) ** 2 + (dHp / Sh) ** 2
                   + Rt * (dCp / Sc) * (dHp / Sh))


# ---------------------------------------------------------------- metrics
def spec_fps(xyz, n, seed=0):
    rng = np.random.default_rng(seed)
    N = len(xyz)
    if N <= n:
        return np.arange(N)
    sel = np.empty(n, np.int64); sel[0] = rng.integers(N)
    d = np.linalg.norm(xyz - xyz[sel[0]], axis=1)
    for i in range(1, n):
        sel[i] = int(d.argmax())
        d = np.minimum(d, np.linalg.norm(xyz - xyz[sel[i]], axis=1))
    return sel


def spec_swd(A, B, n_proj=128, seed=0):
    """Sliced Wasserstein-1. No correspondence — a permutation cannot move it."""
    rng = np.random.default_rng(seed)
    P = rng.normal(size=(A.shape[1], n_proj))
    P /= np.linalg.norm(P, axis=0, keepdims=True)
    a, b = np.sort(A @ P, 0), np.sort(B @ P, 0)
    n = min(len(a), len(b))
    qa = a[np.linspace(0, len(a) - 1, n).round().astype(int)]
    qb = b[np.linspace(0, len(b) - 1, n).round().astype(int)]
    return float(np.abs(qa - qb).mean())


def spec_edge_set(xyz, lab, k=8, pct=90):
    n = len(xyz)
    if n < k + 1:
        return np.zeros(n, bool)
    _, idx = cKDTree(xyz).query(xyz, k=min(k + 1, n))
    contrast = ciede2000(lab[idx[:, 1:]], lab[:, None, :]).max(1)
    return contrast >= np.percentile(contrast, pct)


METRIC_DIR = {"cd_l1": "lower", "cd_l2": "lower", "fscore": "higher",
              "dE00_sym": "lower", "dE00_gt2pred": "lower", "dE00_pred2gt": "lower",
              "dE00_matched": "lower", "match_rate": "higher",
              "colour_swd": "lower", "region_dE": "lower", "colour_edge_iou": "higher"}


def completion_metrics(pred, gt, *, tau=FSCORE_TAU, match_tau=MATCH_TAU, seed=0, n_cells=64):
    """pred/gt: (N,6) — geometry in the PARTIAL's frame, colour as 2*rgb-1.

    Three tiers, because completion has no point correspondence:
      1 geometry            cd_l1, cd_l2, fscore
      2 colour VIA a match  dE00_* (+ match_rate) — partly a geometry metric, by construction
      3 colour WITHOUT one  colour_swd, region_dE, colour_edge_iou — geometry cannot move these
    NaN (never 0) where a metric is undefined.
    """
    pg, pc = pred[:, :3].astype(np.float64), (pred[:, 3:] + 1) / 2
    gg, gc = gt[:, :3].astype(np.float64), (gt[:, 3:] + 1) / 2
    out = {}
    if len(pg) < 4 or len(gg) < 4:
        return {k: np.nan for k in METRIC_DIR}
    tp, tg = cKDTree(pg), cKDTree(gg)
    d_g2p, i_g2p = tp.query(gg, k=1)
    d_p2g, i_p2g = tg.query(pg, k=1)

    out["cd_l1"] = float(d_g2p.mean() + d_p2g.mean())
    out["cd_l2"] = float((d_g2p ** 2).mean() + (d_p2g ** 2).mean())
    prec, rec = float((d_p2g < tau).mean()), float((d_g2p < tau).mean())
    out["fscore"] = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

    lab_g, lab_p = srgb_to_lab(gc), srgb_to_lab(pc)
    transferred = lab_p[i_g2p]
    dE = ciede2000(transferred, lab_g)
    out["dE00_gt2pred"] = float(dE.mean())
    out["dE00_pred2gt"] = float(ciede2000(lab_p, lab_g[i_p2g]).mean())
    out["dE00_sym"] = 0.5 * (out["dE00_gt2pred"] + out["dE00_pred2gt"])
    ok = d_g2p < match_tau
    out["match_rate"] = float(ok.mean())
    out["dE00_matched"] = float(dE[ok].mean()) if ok.sum() >= 8 else np.nan

    out["colour_swd"] = spec_swd(lab_p, lab_g, seed=seed)
    cells = spec_fps(gg, min(n_cells, len(gg)), seed=seed)
    tc = cKDTree(gg[cells])
    cg, cp = tc.query(gg, k=1)[1], tc.query(pg, k=1)[1]
    mg = np.full((len(cells), 3), np.nan); mp = np.full((len(cells), 3), np.nan)
    for c in range(len(cells)):
        a, b = lab_g[cg == c], lab_p[cp == c]
        if len(a): mg[c] = a.mean(0)
        if len(b): mp[c] = b.mean(0)
    both = ~np.isnan(mg).any(1) & ~np.isnan(mp).any(1)
    out["region_dE"] = float(ciede2000(mp[both], mg[both]).mean()) if both.sum() >= 4 else np.nan

    eg, ep = spec_edge_set(gg, lab_g), spec_edge_set(gg, transferred)
    u = (eg | ep).sum()
    out["colour_edge_iou"] = float((eg & ep).sum() / u) if u else np.nan
    return out


EVAL_COLUMNS = ["system", "method", "rep", "obj", "mid", "difficulty", "region",
                "n_pred", "n_gt"] + list(METRIC_DIR)


def spec_score(pred, gt, *, seed=0, equalize_to=None):
    """Score ONE completion. This is the only scoring entry point either notebook may call.

    `equalize_to` FPS-downsamples the prediction before scoring. It exists because the two
    systems under comparison do not generate the same number of points — Joint6D returns the
    N-point cloud it diffused, while the PoinTr+RePaint pipeline returns a union of the visible
    partial and PoinTr's fixed-size completion. Chamfer and F-score both move with point count,
    so comparing them raw would partly measure density rather than quality. Setting
    EQUALIZE_PRED_N puts both at the same count; n_pred is always recorded either way, so the
    choice is visible in the output rather than buried.
    """
    if equalize_to == "gt":
        equalize_to = len(gt)          # match the GT's own count for this object
    if equalize_to and len(pred) > equalize_to:
        pred = pred[spec_fps(pred[:, :3].astype(np.float64), equalize_to, seed=seed)]
    m = completion_metrics(pred, gt, seed=seed)
    m["n_pred"], m["n_gt"] = len(pred), len(gt)
    return m


def spec_config_fingerprint():
    """The config values that MUST match for two runs to be comparable."""
    keys = dict(spec=SPEC_VERSION, category=CATEGORY, synset=SYNSET,
                difficulties=list(DIFFICULTIES), num_points=NUM_POINTS,
                num_train=NUM_TRAIN, num_val=NUM_VAL, seed=SEED,
                fscore_tau=FSCORE_TAU, match_tau=MATCH_TAU,
                equalize_pred_n=EQUALIZE_PRED_N, eval_repeats=EVAL_REPEATS)
    blob = json.dumps(keys, sort_keys=True)
    return keys, hashlib.sha256(blob.encode()).hexdigest()[:16]

'''


import hashlib, json
SPEC_HASH = hashlib.sha256(SPEC_SRC.encode()).hexdigest()[:16]
exec(compile(SPEC_SRC, "<shared_spec>", "exec"), globals())

(RESULTS_DIR / "shared_spec.py").write_text(SPEC_SRC)
CFG_KEYS, CFG_HASH = spec_config_fingerprint()
print(f"SHARED SPEC v{SPEC_VERSION}")
print(f"  code   hash : {SPEC_HASH}")
print(f"  config hash : {CFG_HASH}")
print(f"  {len(METRIC_DIR)} metrics | scoring entry point: spec_score(...)")
print()
for k, v in CFG_KEYS.items():
    print(f"    {k:16s} {v}")
print("\nBoth hashes must match across the notebooks you intend to compare.")
json.dump(dict(spec_hash=SPEC_HASH, cfg_hash=CFG_HASH, cfg=CFG_KEYS, system=SYSTEM_NAME),
          open(RESULTS_DIR / "run_manifest.json", "w"), indent=2)

## §3 · Data — identical objects, identical split, identical frame

Loaded through the shared spec, so this is the *same* set of objects in the *same* order and the
*same* partial-derived frame that System A uses. Any divergence would show up as a differing
`config hash` above.

In [ ]:
t0 = time.time()
DATA_ROOT, SOURCE = spec_fetch_data()
SPLITS, IDS, N_PLY = spec_split(DATA_ROOT)
TRAIN, VAL = SPLITS["train"], SPLITS["val"]
print(f"source : {SOURCE} ({DATA_ROOT})")
print(f"built in {time.time()-t0:.1f}s  |  {len(TRAIN)} train / {len(VAL)} val examples")
print(f"visible fraction: train {np.mean([e['vis'].mean() for e in TRAIN]):.3f}  "
      f"val {np.mean([e['vis'].mean() for e in VAL]):.3f}")


def partial_of(e):
    """(n_vis, 6) — the observed points, in the partial's frame."""
    return e["x0"][e["vis"]]


def missing_of(e):
    """(n_miss, 6) — the ground-truth answer for the region that was removed."""
    return e["x0"][~e["vis"]]


print(f"\nexample: {VAL[0]['mid'][:20]}  n_visible={VAL[0]['vis'].sum()}  "
      f"n_missing={(~VAL[0]['vis']).sum()}")
print("\nCIEDE2000 self-test:", end=" ")
_T1 = np.array([[50, 2.6772, -79.7751], [50, 0, 0], [50, 2.5, 0]])
_T2 = np.array([[50, 0, -82.7485], [50, -1, 2], [50, 0, -2.5]])
assert np.allclose(ciede2000(_T1, _T2), [2.0425, 2.3669, 4.3065], atol=1e-3)
print("PASS")

## §4 · Geometry stage — frozen PoinTr

Ported from `scripts/run_benchmark_kaggle.py` with the same checkpoint, the same config
(`trans_dim=384, knn_layer=1, num_pred=6144, num_query=96`) and the same pure-PyTorch
`pointnet2_ops` shim, so no CUDA extension has to compile.

`assert len(missing) == 0` on the state-dict load is the guard that matters: it fails loudly if the
checkpoint is not the original PoinTr, rather than silently evaluating a partly-random network.

If any of this fails, set `GEOMETRY_SOURCE = "gt_oracle"` in §1. The colour comparison remains
completely valid — it simply becomes a comparison at *perfect* geometry, which is a useful
measurement in its own right rather than a degraded one.

In [ ]:
GEO_READY = False
if GEOMETRY_SOURCE == "pointr":
    try:
        # ---- pure-torch shim for pointnet2_ops (PoinTr uses exactly these six) ----
        import types
        def furthest_point_sample(xyz, npoint):
            B, N, _ = xyz.shape
            idx = torch.zeros(B, npoint, dtype=torch.long, device=xyz.device)
            dist = torch.full((B, N), 1e10, device=xyz.device, dtype=xyz.dtype)
            far = torch.zeros(B, dtype=torch.long, device=xyz.device)
            ar = torch.arange(B, device=xyz.device)
            for i in range(npoint):
                idx[:, i] = far
                dist = torch.minimum(dist, ((xyz - xyz[ar, far].unsqueeze(1)) ** 2).sum(-1))
                far = torch.max(dist, dim=1).indices
            return idx.int()

        def gather_operation(features, idx):
            B, C, N = features.shape; idx = idx.long()
            return torch.gather(features, 2, idx.unsqueeze(1).expand(B, C, idx.shape[1])).contiguous()

        def three_nn(query, ref):
            d = torch.cdist(query, ref)
            dist, idx = d.topk(3, dim=-1, largest=False)
            return dist.contiguous(), idx.int().contiguous()

        def three_interpolate(features, idx, weight):
            B, C, M = features.shape; N = idx.shape[1]
            i = idx.long().view(B, 1, N * 3).expand(B, C, N * 3)
            return (torch.gather(features, 2, i).view(B, C, N, 3)
                    * weight.unsqueeze(1)).sum(-1).contiguous()

        def grouping_operation(features, idx):
            B, C, N = features.shape; _, S, K = idx.shape
            i = idx.long().view(B, 1, S * K).expand(B, C, S * K)
            return torch.gather(features, 2, i).view(B, C, S, K).contiguous()

        def ball_query(radius, nsample, xyz, new_xyz):
            d = torch.cdist(new_xyz, xyz)
            idx = d.argsort(dim=-1)[:, :, :nsample]
            first = idx[:, :, :1].expand_as(idx)
            return torch.where(torch.gather(d, 2, idx) < radius, idx, first).int().contiguous()

        m = types.ModuleType("pointnet2_ops")
        u = types.ModuleType("pointnet2_ops.pointnet2_utils")
        for f in (furthest_point_sample, gather_operation, three_nn, three_interpolate,
                  grouping_operation, ball_query):
            setattr(u, f.__name__, f)
        m.pointnet2_utils = u
        sys.modules["pointnet2_ops"] = m
        sys.modules["pointnet2_ops.pointnet2_utils"] = u

        # ---- stub the compiled LOSS extensions ------------------------------------------
        # models/PoinTr.py does `from extensions.chamfer_dist import ChamferDistanceL1/L2`
        # at module level, and extensions/chamfer_dist/__init__.py does `import chamfer`.
        # Those are TRAINING losses; inference never calls them. Rather than compiling CUDA
        # extensions against a torch build they predate, we inject stubs that raise loudly if
        # anything ever actually calls them — so a silent wrong answer is impossible.
        def _never(name):
            def _f(*a, **k):
                raise NotImplementedError(
                    f"{name} is a stub: this notebook only runs PoinTr in INFERENCE mode. "
                    "If you see this, something is trying to compute a training loss.")
            return _f

        for _ext in ("chamfer", "gridding", "gridding_distance", "cubic_feature_sampling",
                     "emd"):
            if _ext not in sys.modules:
                _m = types.ModuleType(_ext)
                for _fn in ("forward", "backward"):
                    setattr(_m, _fn, _never(f"{_ext}.{_fn}"))
                sys.modules[_ext] = _m

        POINTR = WORK / "PoinTr"
        if not POINTR.is_dir():
            subprocess.run(f"git clone -q {POINTR_REPO} {POINTR}", shell=True, check=True)
        sys.path.insert(0, str(POINTR))

        CKPT = WORK / "PoinTr_ShapeNet55.pth"
        if not CKPT.exists():
            for c in glob.glob("/kaggle/input/**/PoinTr_ShapeNet55.pth", recursive=True):
                shutil.copy(c, CKPT); break
        if not CKPT.exists():
            subprocess.run(f"pip install -q gdown && gdown -q {POINTR_CKPT_ID} -O {CKPT}",
                           shell=True, check=True)

        from easydict import EasyDict
        from models.PoinTr import PoinTr, fps
        cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
        geo = PoinTr(cfg)
        sd = torch.load(CKPT, map_location="cpu", weights_only=False)
        base = sd.get("base_model", sd.get("model", sd))
        base = {k.replace("module.", ""): v for k, v in base.items()}
        mi, ui = geo.load_state_dict(base, strict=False)
        assert len(mi) == 0, f"not the original PoinTr — missing keys {mi[:4]}"
        for p in geo.parameters():
            p.requires_grad_(False)
        geo.eval().to(DEV)

        # smoke the forward pass NOW, not 20 minutes into the run
        with torch.no_grad():
            _p = torch.randn(1, 512, 3, device=DEV) * 0.3
            _o = geo(_p)[1]
        assert _o.shape[1] >= geo.num_pred, f"unexpected PoinTr output {tuple(_o.shape)}"
        assert torch.isfinite(_o).all(), "PoinTr produced non-finite output"
        GEO_READY = True
        print(f"frozen PoinTr ready | num_pred={geo.num_pred} | unexpected keys: {len(ui)}")
        print(f"forward smoke test: (1,512,3) -> {tuple(_o.shape)}, all finite  PASS")
    except Exception as ex:
        import traceback
        print(f"!! PoinTr setup FAILED: {type(ex).__name__}: {ex}\n")
        traceback.print_exc()
        if not ALLOW_GEOMETRY_FALLBACK:
            raise RuntimeError(
                "\n\n" + "=" * 78 +
                "\nPoinTr did not load, and GEOMETRY_SOURCE='pointr' was requested.\n"
                "Stopping instead of silently substituting PERFECT ground-truth geometry —\n"
                "that fallback produces cd_l1=0 / fscore=1.0 rows that look like results and\n"
                "are not. Either fix the error above, or set ALLOW_GEOMETRY_FALLBACK=True in\n"
                "§1 to deliberately run the colour-only comparison.\n" + "=" * 78) from ex
        print("!! ALLOW_GEOMETRY_FALLBACK is set -> falling back to 'gt_oracle'.")
        print("!! The COLOUR comparison stays valid, but geometry is PERFECT and its numbers")
        print("!! must NOT be compared against System A's.")
        GEOMETRY_SOURCE = "gt_oracle"

if GEOMETRY_SOURCE == "gt_oracle":
    print("geometry stage: GT ORACLE (perfect geometry; isolates the colouring method)")

### §5 · Frame alignment

PoinTr was trained on ShapeNet-55, whose canonical axes differ from ShapeNet-Part's. The original
harness resolves this by **measuring**: all 6 axis permutations × 8 sign flips are scored by Chamfer
against the ground truth on a few training objects, and the best is kept. That is reproduced here
rather than hard-coding an orientation, and the full ranking is printed so a near-tie is visible.

In [ ]:
AXIS_PERM, AXIS_SIGN = (2, 1, 0), (1, 1, 1)


def _fps_np(xyz, n):
    if len(xyz) <= n:
        return np.ascontiguousarray(xyz, np.float32)
    t = torch.from_numpy(np.ascontiguousarray(xyz[:, :3])).float().unsqueeze(0).to(DEV)
    idx = furthest_point_sample(t, n)[0].long().cpu().numpy()
    return np.ascontiguousarray(xyz[idx], np.float32)


@torch.no_grad()
def complete_geometry(partial_xyz, perm=None, sign=None):
    """Colourless partial -> PoinTr completion, mapped into the frame and back out again."""
    perm = AXIS_PERM if perm is None else tuple(perm)
    sign = np.asarray(AXIS_SIGN if sign is None else sign, np.float32)
    x = np.ascontiguousarray(partial_xyz[:, :3][:, list(perm)] * sign)
    p = torch.from_numpy(x).float().unsqueeze(0).to(DEV)
    fine = geo(p)[1][0, :geo.num_pred].cpu().numpy()
    inv = np.argsort(perm)
    return np.ascontiguousarray(fine[:, inv] * sign[inv])


def chamfer_l1(a, b):
    return float(cKDTree(b).query(a, k=1)[0].mean() + cKDTree(a).query(b, k=1)[0].mean())


def find_frame(entries, n_probe=FRAME_PROBE):
    """48 candidates, scored by Chamfer against GT. Measured, not eyeballed."""
    import itertools
    rows = []
    for perm in itertools.permutations(range(3)):
        for sign in itertools.product([1, -1], repeat=3):
            s = 0.0
            for e in entries[:n_probe]:
                comp = complete_geometry(partial_of(e)[:, :3], perm, sign)
                s += chamfer_l1(_fps_np(comp, COMP_POINTS), e["x0"][:, :3])
            rows.append((s / n_probe, perm, sign))
    rows.sort()
    return rows


if GEO_READY:
    t0 = time.time()
    RANK = find_frame(TRAIN)
    AXIS_PERM, AXIS_SIGN = RANK[0][1], RANK[0][2]
    print(f"axis search over 48 candidates on {FRAME_PROBE} objects ({time.time()-t0:.0f}s)\n")
    print(f"{'rank':>4} {'chamfer_l1':>12}  perm        sign")
    for i, (c, p, s) in enumerate(RANK[:5]):
        print(f"{i+1:>4} {c:>12.5f}  {p}  {s}")
    print(f"{'':>4} {'...':>12}")
    print(f"{len(RANK):>4} {RANK[-1][0]:>12.5f}  {RANK[-1][1]}  {RANK[-1][2]}   (worst)")
    print(f"\nchosen: AXIS_PERM={AXIS_PERM} AXIS_SIGN={AXIS_SIGN}")
    _margin = RANK[1][0] / RANK[0][0] - 1
    print(f"margin over runner-up: {_margin:.1%}"
          + ("" if _margin > .05 else "   <-- NARROW; the frame is not clearly determined"))
    pd.DataFrame([dict(rank=i+1, chamfer_l1=c, perm=str(p), sign=str(s))
                  for i, (c, p, s) in enumerate(RANK)]).to_csv(
        RESULTS_DIR / "tables/axis_search.csv", index=False)
else:
    RANK = None
    print("skipped (no PoinTr)")

In [ ]:
def completed_geometry_for(e):
    """The geometry every colouring method in this notebook is scored on.

    All six methods share it, which is the design's real strength: differences between them are
    differences in COLOURING, with the completion held fixed.
    """
    if GEOMETRY_SOURCE == "gt_oracle":
        return missing_of(e)[:, :3].copy()
    comp = complete_geometry(partial_of(e)[:, :3])
    return _fps_np(comp, COMP_POINTS)


if RUN_FULL_EXPERIMENT or GEOMETRY_SOURCE == "gt_oracle":
    t0 = time.time()
    COMP = {}
    for i, e in enumerate(VAL):
        COMP[i] = completed_geometry_for(e)
        if i == 0:
            print(f"first completion: {COMP[0].shape[0]} points "
                  f"(GT missing region has {(~e['vis']).sum()})")
    print(f"completed {len(COMP)} validation objects in {time.time()-t0:.1f}s")
    _cd = np.mean([chamfer_l1(COMP[i], missing_of(VAL[i])[:, :3]) for i in range(len(VAL))])
    print(f"\nmean chamfer_l1 of the completion vs the GT missing region: {_cd:.5f}")
    if GEOMETRY_SOURCE == "gt_oracle":
        print("(0 by construction — geometry is the ground truth in this mode)")
else:
    COMP = None
    print("skipped — set RUN_FULL_EXPERIMENT = True")

## §6 · Part segmentation

The same PointNet part-seg as the original harness, trained on GT xyz → GT part labels. Its accuracy
is the binding constraint on every part-conditioned method below: **if the labels are poor, the
part prior cannot help no matter how good the colouring rule is.** That is why
`RePaint-part (oracle seg)` is scored separately — it is the same colour model given perfect labels,
which separates *"is the part idea good?"* from *"is the segmenter good enough?"*.

Accuracy is reported on the **validation** objects the segmenter never saw, not on its training set.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class _TNet(nn.Module):
    def __init__(s, k):
        super().__init__(); s.k = k
        s.mlp = nn.Sequential(nn.Conv1d(k, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
                              nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
                              nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU())
        s.fc = nn.Sequential(nn.Linear(1024, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU(),
                             nn.Linear(256, k * k))

    def forward(s, x):
        B = x.size(0); f = s.mlp(x).max(-1)[0]
        return s.fc(f).view(B, s.k, s.k) + torch.eye(s.k, device=x.device).unsqueeze(0)


class PointNetPartSeg(nn.Module):
    def __init__(s, P):
        super().__init__(); s.itn = _TNet(3)
        s.mlp1 = nn.Sequential(nn.Conv1d(3, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
                               nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        s.fstn = _TNet(128)
        s.mlp2 = nn.Sequential(nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
                               nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU())
        s.seg = nn.Sequential(nn.Conv1d(1152, 512, 1), nn.BatchNorm1d(512), nn.ReLU(),
                              nn.Conv1d(512, 256, 1), nn.BatchNorm1d(256), nn.ReLU(),
                              nn.Conv1d(256, P, 1))

    def forward(s, x):
        x = x.transpose(1, 2); x = torch.bmm(s.itn(x), x)
        f = s.mlp1(x); f = torch.bmm(s.fstn(f), f); pf = f
        g = s.mlp2(f).max(-1, keepdim=True)[0].expand(-1, -1, f.size(-1))
        return s.seg(torch.cat([pf, g], 1)).transpose(1, 2)


def _unit(x):
    x = np.asarray(x, np.float32)[:, :3]
    c = x.mean(0)
    return (x - c) / (np.linalg.norm(x - c, axis=1).max() + 1e-9)


@torch.no_grad()
def segment(model, xyz):
    model.eval()
    t = torch.from_numpy(_unit(xyz)).float().unsqueeze(0).to(DEV)
    return model(t)[0].argmax(-1).cpu().numpy()


def segment_oracle(e, xyz):
    """Labels from the nearest GT point — the segmentation-error-free ceiling."""
    g = _unit(e["x0"][:, :3])
    return e["part"][cKDTree(g).query(_unit(xyz), k=1)[1]]


class _PartDS(Dataset):
    def __init__(s, E): s.E = E
    def __len__(s): return len(s.E)
    def __getitem__(s, i):
        e = s.E[i]
        return torch.from_numpy(_unit(e["x0"][:, :3])).float(), torch.from_numpy(e["part"]).long()


def train_partseg(epochs=SEG_EPOCHS):
    torch.manual_seed(SEED)
    m = PointNetPartSeg(NUM_PARTS).to(DEV)
    dl = DataLoader(_PartDS(TRAIN), batch_size=16, shuffle=True, drop_last=len(TRAIN) > 16)
    opt = torch.optim.Adam(m.parameters(), SEG_LR); lf = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train(); cor = seen = 0; tot = 0.0
        for xyz, lab in dl:
            xyz, lab = xyz.to(DEV), lab.to(DEV)
            opt.zero_grad()
            lo = m(xyz); loss = lf(lo.reshape(-1, lo.size(-1)), lab.reshape(-1))
            loss.backward(); opt.step()
            tot += loss.item() * xyz.size(0)
            cor += (lo.argmax(-1) == lab).sum().item(); seen += lab.numel()
        if ep % 10 == 0 or ep == epochs - 1:
            print(f"  ep {ep:3d}  loss {tot/len(dl.dataset):.4f}  train acc {cor/seen*100:.1f}%")
    return m


if RUN_FULL_EXPERIMENT:
    ck = RESULTS_DIR / "checkpoints/partseg.pt"
    SEG = PointNetPartSeg(NUM_PARTS).to(DEV)
    if ck.exists():
        SEG.load_state_dict(torch.load(ck, map_location=DEV, weights_only=False))
        print("part-seg loaded from checkpoint")
    else:
        print(f"training part-seg, {SEG_EPOCHS} epochs\n")
        SEG = train_partseg(); torch.save(SEG.state_dict(), ck)

    accs = [float((segment(SEG, e["x0"][:, :3]) == e["part"]).mean()) for e in VAL]
    SEG_ACC = float(np.mean(accs))
    print(f"\nHELD-OUT part-seg accuracy: {SEG_ACC:.1%}  "
          f"(per-object min {min(accs):.1%} / max {max(accs):.1%})")
    print("This number governs how much the part-conditioned methods can possibly gain.")
    pd.DataFrame(dict(mid=[e["mid"] for e in VAL], acc=accs)).to_csv(
        RESULTS_DIR / "tables/segmentation_accuracy.csv", index=False)
else:
    SEG, SEG_ACC = None, np.nan
    print("skipped — set RUN_FULL_EXPERIMENT = True")

## §7 · Colour DDPM — the part-conditioned denoiser

Ported unchanged from the harness: EdgeConv over an XYZ kNN graph, a **part-mean pool** broadcast
back to every point, and a global pool, all FiLM-modulated by $t$. `part_cond=False` removes the
part one-hot and the part pool, giving the `RePaint-vanilla` ablation.

One property worth stating plainly, because it is a real difference from System A: **this model is
trained unconditionally on complete clouds and never sees an occlusion mask.** The mask enters only
at inference, through RePaint. System A trains with the mask in the loop.

Relation to the feasibility findings: `part_pool` is *additive* context, which maps onto Question B2
(**neutral**, all metrics unresolved) — not onto the *restrictive* same-part masking of Question C,
which was harmful (win rate 2.6 %). So the prediction going in is that `RePaint-part` ≈
`RePaint-vanilla`, with the part machinery paying complexity for little. The oracle-seg arm tests
whether poor labels are the reason.

In [ ]:
def _gather_nb(h, idx):
    B, N, C = h.shape
    off = (torch.arange(B, device=h.device) * N).view(B, 1, 1)
    return h.reshape(B * N, C)[(idx + off).reshape(-1)].reshape(B, N, idx.shape[-1], C)


def knn_graph(xyz, k, chunk=2048):
    B, N, _ = xyz.shape
    out = torch.empty(B, N, k, dtype=torch.long, device=xyz.device)
    kk = min(k + 1, N)
    for s in range(0, N, chunk):
        d = torch.cdist(xyz[:, s:s + chunk], xyz)
        idx = d.topk(kk, dim=-1, largest=False).indices[:, :, 1:]
        if idx.shape[-1] < k:
            idx = idx[..., [i % idx.shape[-1] for i in range(k)]]
        out[:, s:s + chunk] = idx
    return out


def part_pool(h, part, P):
    """Within-part mean, broadcast back to every point. THIS is where 'part-based' lives:
    a part's visible colours drive its own missing points."""
    B, N, C = h.shape
    flat = (part + torch.arange(B, device=h.device).view(B, 1) * P).reshape(-1)
    s = torch.zeros(B * P, C, device=h.device, dtype=h.dtype).index_add_(0, flat, h.reshape(-1, C))
    n = torch.zeros(B * P, 1, device=h.device, dtype=h.dtype).index_add_(
        0, flat, torch.ones(B * N, 1, device=h.device, dtype=h.dtype))
    return torch.gather((s / n.clamp(min=1.0)).reshape(B, P, C), 1,
                        part.unsqueeze(-1).expand(B, N, C))


def timestep_embedding(t, dim):
    """Sinusoidal embedding. For ODD dim, sin+cos give 2*(dim//2) = dim-1 channels, so the
    result is padded — without this an odd width fails with a shape error inside the first
    Linear, and only once training actually starts."""
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float().view(-1, 1) * f.view(1, -1)
    e = torch.cat([a.sin(), a.cos()], -1)
    return F.pad(e, (0, dim - e.shape[-1])) if e.shape[-1] < dim else e


class Block(nn.Module):
    def __init__(self, w, part_cond=True):
        super().__init__(); self.part_cond = part_cond
        self.edge = nn.Sequential(nn.Linear(2 * w + 4, w), nn.GELU(), nn.Linear(w, w))
        ctx = w * (3 if part_cond else 2)
        self.fuse = nn.Sequential(nn.LayerNorm(ctx), nn.Linear(ctx, w), nn.GELU(), nn.Linear(w, w))
        self.film = nn.Linear(w, 2 * w)

    def forward(self, h, idx, rel, part, P, temb):
        hj = _gather_nb(h, idx); hi = h.unsqueeze(2).expand_as(hj)
        e = self.edge(torch.cat([hi, hj - hi, rel], -1)).max(2).values
        g = h.max(1, keepdim=True).values.expand_as(h)
        c = [e, g] + ([part_pool(h, part, P)] if self.part_cond else [])
        d = self.fuse(torch.cat(c, -1))
        sc, sh = self.film(temb).unsqueeze(1).chunk(2, -1)
        return h + d * (1 + sc) + sh


class PartColorDenoiser(nn.Module):
    """eps-prediction on a per-point COLOUR field, conditioned on xyz (+ part)."""

    def __init__(self, num_parts, width=DDPM_WIDTH, k=DDPM_K, n_blocks=DDPM_BLOCKS, part_cond=True):
        super().__init__()
        self.P, self.k, self.part_cond, self.width = num_parts, k, part_cond, width
        self.inp = nn.Linear(3 + 3 + (num_parts if part_cond else 0), width)
        self.temb = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList([Block(width, part_cond) for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Linear(width, 3))

    def build_ctx(self, xyz, part):
        """Geometry is FIXED across the reverse chain, so the graph is built once — a large
        speed-up, and it guarantees every step sees identical topology."""
        idx = knn_graph(xyz, self.k)
        rel = _gather_nb(xyz, idx) - xyz.unsqueeze(2)
        sc = rel.norm(dim=-1).mean((1, 2), keepdim=True).clamp(min=1e-6).unsqueeze(-1)
        rel = torch.cat([rel / sc, rel.norm(dim=-1, keepdim=True) / sc], -1)
        return dict(xyz=xyz, idx=idx, rel=rel, part=part,
                    onehot=F.one_hot(part, self.P).float())

    def forward(self, c_t, t, ctx):
        B = c_t.shape[0]
        f = [ctx["xyz"], c_t] + ([ctx["onehot"]] if self.part_cond else [])
        h = self.inp(torch.cat(f, -1))
        temb = self.temb(timestep_embedding(
            t.expand(B) if t.dim() == 0 else t.repeat(B) if t.numel() == 1 else t, self.width))
        for b in self.blocks:
            h = b(h, ctx["idx"], ctx["rel"], ctx["part"], self.P, temb)
        return self.out(h)


def _npar(**kw):
    return sum(p.numel() for p in PartColorDenoiser(NUM_PARTS, **kw).parameters())


# The part-conditioned model is intrinsically LARGER: the part one-hot adds input channels and
# part_pool adds a third context branch. Comparing it against the part-blind model at its native
# width would confound "does the part prior help?" with "is the bigger model better?".
# So the part-BLIND model is widened until the two match.
TARGET = _npar(part_cond=True)
VAN_WIDTH = DDPM_WIDTH
if MATCH_DDPM_PARAMS:
    # All widths, odd included: timestep_embedding pads odd dims (asserted below), and an odd
    # width matches the parameter budget far more tightly here (135 -> 0.07% vs 136 -> 1.41%).
    cands = range(16, 4 * DDPM_WIDTH + 1)
    VAN_WIDTH = min(cands, key=lambda w: abs(_npar(part_cond=False, width=w) - TARGET))
    assert timestep_embedding(torch.zeros(1), VAN_WIDTH).shape[-1] == VAN_WIDTH, \
        f"timestep embedding is not {VAN_WIDTH} channels wide"

_np_part, _np_van = _npar(part_cond=True), _npar(part_cond=False, width=VAN_WIDTH)
print(f"part-conditioned : width {DDPM_WIDTH}  {_np_part:,} params")
print(f"part-blind       : width {VAN_WIDTH}  {_np_van:,} params")
print(f"gap              : {abs(_np_van-_np_part)/_np_part:+.2%}"
      + ("   (matched — P2 is an architecture comparison, not a capacity one)"
         if MATCH_DDPM_PARAMS else
         "   <-- NOT matched; P2 confounds the part prior with model size"))
if MATCH_DDPM_PARAMS:
    assert abs(_np_van - _np_part) / _np_part < 0.03, "width search failed to match"

In [ ]:
def cosine_betas_c(T, s=0.008):
    t = torch.linspace(0, T, T + 1) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    ab = f / f[0]
    return (1 - ab[1:] / ab[:-1]).clamp(1e-8, 0.999)


class ColourDiffusion:
    """Plain DDPM (eps-prediction) on an (N,3) colour field in [-1,1]."""

    def __init__(self, T=DDPM_T, device=DEV):
        self.T, self.device = T, device
        b = cosine_betas_c(T).to(device); a = 1.0 - b
        abar = torch.cumprod(a, 0)
        abar_prev = torch.cat([torch.ones(1, device=device), abar[:-1]])
        self.betas, self.alphas, self.abar, self.abar_prev = b, a, abar, abar_prev
        self.sqrt_abar, self.sqrt_1mabar = abar.sqrt(), (1 - abar).sqrt()
        self.post_var = b * (1 - abar_prev) / (1 - abar)
        self.post_c0 = b * abar_prev.sqrt() / (1 - abar)
        self.post_ct = (1 - abar_prev) * a.sqrt() / (1 - abar)

    def q_sample(self, x0, t, noise=None):
        noise = torch.randn_like(x0) if noise is None else noise
        sa = self.sqrt_abar[t].view(-1, *([1] * (x0.dim() - 1)))
        sb = self.sqrt_1mabar[t].view(-1, *([1] * (x0.dim() - 1)))
        return sa * x0 + sb * noise

    def p_sample(self, eps, x_t, t, generator=None):
        x0 = ((x_t - self.sqrt_1mabar[t] * eps) / self.sqrt_abar[t]).clamp(-1, 1)
        mean = self.post_c0[t] * x0 + self.post_ct[t] * x_t
        if t == 0:
            return mean, x0
        z = torch.randn(x_t.shape, device=x_t.device, dtype=x_t.dtype, generator=generator)
        return mean + self.post_var[t].sqrt() * z, x0

    def forward_jump(self, x, t, generator=None):
        z = torch.randn(x.shape, device=x.device, dtype=x.dtype, generator=generator)
        return self.alphas[t].sqrt() * x + self.betas[t].sqrt() * z


def get_schedule_jump(T, jump_length=JUMP_LENGTH, jump_n_sample=JUMP_N_SAMPLE):
    """RePaint's resampling ('time-travel') schedule — descending pair = reverse step,
    ascending pair = forward jump."""
    jumps = {j: jump_n_sample - 1 for j in range(0, T - jump_length, jump_length)}
    t, ts = T, []
    while t >= 1:
        t -= 1; ts.append(t)
        if jumps.get(t, 0) > 0:
            jumps[t] -= 1
            for _ in range(jump_length):
                t += 1; ts.append(t)
    ts.append(-1)
    return ts


CDIF = ColourDiffusion()
print(f"colour diffusion: T={CDIF.T} | RePaint steps at "
      f"(j={JUMP_LENGTH}, U={JUMP_N_SAMPLE}): {len(get_schedule_jump(CDIF.T))}")


def train_colour_ddpm(part_cond, epochs=DDPM_EPOCHS, tag=""):
    """Unconditional training on COMPLETE clouds — the occlusion mask is never seen here."""
    torch.manual_seed(SEED)
    w = DDPM_WIDTH if part_cond else VAN_WIDTH
    m = PartColorDenoiser(NUM_PARTS, width=w, part_cond=part_cond).to(DEV)
    ck = RESULTS_DIR / f"checkpoints/ddpm_{tag}.pt"
    cfg = (epochs, w, DDPM_BLOCKS, DDPM_K, len(TRAIN), SEED)
    if ck.exists():
        st = torch.load(ck, map_location=DEV, weights_only=False)
        # a bare state_dict is a checkpoint from BEFORE parameter matching; its width may differ,
        # which would raise a shape error mid-load. Check the config instead of trusting the name.
        if isinstance(st, dict) and st.get("cfg") == cfg:
            m.load_state_dict(st["model"])
            print(f"  [{tag}] loaded from checkpoint (width {w})"); return m
        print(f"  [{tag}] checkpoint exists but its config differs "
              f"(likely width {DDPM_WIDTH} from a pre-parameter-matching run) -> retraining")
    opt = torch.optim.AdamW(m.parameters(), DDPM_LR, weight_decay=1e-4)
    X = torch.from_numpy(np.stack([e["x0"][:, :3] for e in TRAIN])).float()
    C = torch.from_numpy(np.stack([e["x0"][:, 3:] for e in TRAIN])).float()
    P = torch.from_numpy(np.stack([e["part"] for e in TRAIN])).long()
    g = torch.Generator().manual_seed(SEED)

    # resume from a partial run: a dropped session should cost minutes, not the whole stage
    part_ck = RESULTS_DIR / f"checkpoints/ddpm_{tag}_partial.pt"
    start = 0
    if part_ck.exists():
        st = torch.load(part_ck, map_location=DEV, weights_only=False)
        if st.get("cfg") == (epochs, w, DDPM_BLOCKS, DDPM_K, len(TRAIN), SEED):
            m.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); start = st["epoch"] + 1
            print(f"  [{tag}] resumed at epoch {start}")
    for _ in range(start * max(1, len(X) // DDPM_BS)):
        torch.randint(len(X), (DDPM_BS,), generator=g)      # keep the batch stream aligned

    for ep in range(start, epochs):
        tot = n = 0
        for _ in range(max(1, len(X) // DDPM_BS)):
            bi = torch.randint(len(X), (DDPM_BS,), generator=g)
            xyz, c0, prt = X[bi].to(DEV), C[bi].to(DEV), P[bi].to(DEV)
            ctx = m.build_ctx(xyz, prt)
            t = torch.randint(0, CDIF.T, (DDPM_BS,), device=DEV)
            eps = torch.randn_like(c0)
            loss = F.mse_loss(m(CDIF.q_sample(c0, t, eps), t, ctx), eps)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        if ep % 25 == 0 or ep == epochs - 1:
            torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), epoch=ep,
                            cfg=(epochs, w, DDPM_BLOCKS, DDPM_K, len(TRAIN), SEED)),
                       part_ck)
        if ep % 50 == 0 or ep == epochs - 1:
            print(f"  [{tag}] ep {ep:3d}/{epochs}  loss {tot/n:.4f}", flush=True)
    torch.save(dict(model=m.state_dict(), cfg=cfg), ck)
    part_ck.unlink(missing_ok=True)
    return m


# forward/backward smoke on BOTH models, at the exact widths that will be trained.
# The previous run died 19 minutes in on a shape error that this catches in under a second.
for _pc, _w in ((True, DDPM_WIDTH), (False, VAN_WIDTH)):
    _m = PartColorDenoiser(NUM_PARTS, width=_w, part_cond=_pc).to(DEV)
    _x = torch.randn(2, 64, 3, device=DEV)
    _p = torch.randint(0, NUM_PARTS, (2, 64), device=DEV)
    _c = torch.randn(2, 64, 3, device=DEV)
    _t = torch.randint(0, CDIF.T, (2,), device=DEV)
    _o = _m(_c, _t, _m.build_ctx(_x, _p))
    assert _o.shape == _c.shape, f"part_cond={_pc} width={_w}: output {tuple(_o.shape)}"
    F.mse_loss(_o, _c).backward()
    print(f"  smoke  part_cond={str(_pc):5s} width {_w:3d} -> {tuple(_o.shape)}  fwd+bwd OK")
del _m, _x, _p, _c, _t, _o
print()

if RUN_FULL_EXPERIMENT:
    print(f"training two colour DDPMs, {DDPM_EPOCHS} epochs each\n")
    t0 = time.time()
    DDPM_PART = train_colour_ddpm(True, tag="part")
    DDPM_VAN = train_colour_ddpm(False, tag="vanilla")
    print(f"\ndone in {(time.time()-t0)/60:.1f} min")
else:
    DDPM_PART = DDPM_VAN = None
    print("skipped — set RUN_FULL_EXPERIMENT = True")

## §8 · RePaint inference and the six colouring methods

Faithful to Lugmayr et al. (2022): at every reverse step the **known** colours are re-injected at
the correct noise level, and the schedule jumps back up so the generated region can harmonise with
them rather than merely matching its texture.

$$x_{t-1} = m \odot q(x_{\text{known}}, t{-}1) \;+\; (1-m)\odot p_\theta(x_t)$$

Three things make it *part-based*, all preserved from the original: features are pooled **within a
part**; a part with zero visible points is flagged **anchorless** and filled from the learned prior
rather than bleeding colour from a neighbouring part; and the resampling budget adapts to the
worst-covered part.

**One deliberate deviation from the original harness.** It re-centred the union cloud to its own
unit sphere at inference while training on differently-normalised clouds. Here training and
inference both use the shared partial frame, which removes that train/test mismatch. This makes the
pipeline slightly *better* than the original, not worse — worth knowing when comparing against
previously published numbers from this repo.

In [ ]:
@torch.no_grad()
def repaint_colours(model, xyz, part, known, known_col, *, seed=0, return_diag=False):
    """xyz (N,3) partial-frame; part (N,); known (N,) bool; known_col (N,3) in [-1,1].
    Returns (N,3) in [-1,1]."""
    g = torch.Generator(device=DEV).manual_seed(seed)
    xyz_t = torch.as_tensor(xyz).float().unsqueeze(0).to(DEV)
    prt = torch.as_tensor(part).long().unsqueeze(0).to(DEV)
    m = torch.as_tensor(known).bool().view(1, -1, 1).to(DEV)
    c0 = torch.as_tensor(known_col).float().unsqueeze(0).to(DEV) * m

    pid = prt[0]
    vis = np.array([(m[0, :, 0][pid == p].float().mean().item() if (pid == p).any() else np.nan)
                    for p in range(model.P)])
    present = ~np.isnan(vis)
    anchorless = [p for p in range(model.P) if present[p] and vis[p] == 0.0]
    U = JUMP_N_SAMPLE
    if ADAPTIVE_JUMPS and present.any():
        worst = float(np.nanmin(np.where(present, vis, np.nan)))
        U = int(np.clip(round(JUMP_N_SAMPLE * (1.5 - worst)), 1, 2 * JUMP_N_SAMPLE))

    model.eval().to(DEV)
    ctx = model.build_ctx(xyz_t, prt)
    x = torch.randn(1, xyz_t.shape[1], 3, device=DEV, generator=g)
    ts = get_schedule_jump(CDIF.T, JUMP_LENGTH, U)
    for t_cur, t_next in zip(ts[:-1], ts[1:]):
        if t_next < t_cur:
            eps = model(x, torch.tensor(t_cur, device=DEV), ctx)
            x_unknown, _ = CDIF.p_sample(eps, x, t_cur, generator=g)
            if t_cur > 0:
                noise = torch.randn(x.shape, device=DEV, generator=g)
                x_known = CDIF.q_sample(c0, torch.tensor([t_cur - 1], device=DEV), noise)
            else:
                x_known = c0
            x = torch.where(m, x_known, x_unknown)
        else:
            x = CDIF.forward_jump(x, t_cur, generator=g)

    out = x[0].clamp(-1, 1).cpu().numpy()
    kn = np.asarray(known, bool)
    out[kn] = np.asarray(known_col, np.float32)[kn]        # visible colours preserved exactly
    if return_diag:
        return out, dict(part_visibility=vis, anchorless=anchorless, U=U, n_steps=len(ts))
    return out


def build_union(e, comp_xyz):
    """visible partial (colour KNOWN) + completed geometry (colour UNKNOWN), partial frame."""
    p = partial_of(e)
    xyz = np.concatenate([p[:, :3], comp_xyz], 0).astype(np.float32)
    known = np.zeros(len(xyz), bool); known[:len(p)] = True
    col = np.zeros((len(xyz), 3), np.float32); col[:len(p)] = p[:, 3:]
    return xyz, known, col, len(p)

In [ ]:
def colour_completion(e, comp_xyz, method, seed=0):
    """-> (n_comp, 6): the completed points with this method's colour, partial-frame units.

    Every method is handed IDENTICAL geometry, so differences are differences in colouring.
    """
    xyz, known, col, n_vis = build_union(e, comp_xyz)
    p = partial_of(e)

    if method == "NN-copy":
        i = cKDTree(p[:, :3]).query(comp_xyz, k=1)[1]
        c = p[i, 3:]

    elif method == "part-mean":
        vl, cl = segment(SEG, p[:, :3]), segment(SEG, comp_xyz)
        mean = np.tile(p[:, 3:].mean(0), (NUM_PARTS, 1))       # unsupported part -> global mean
        for k in range(NUM_PARTS):
            m_ = vl == k
            if m_.any():
                mean[k] = p[m_, 3:].mean(0)
        c = mean[cl]

    elif method == "oracle ceiling":
        # per-part GROUND-TRUTH mean colour: the best any part-constant rule could do
        gp, gc = e["part"], e["x0"][:, 3:]
        mean = np.tile(gc.mean(0), (NUM_PARTS, 1))
        for k in range(NUM_PARTS):
            m_ = gp == k
            if m_.any():
                mean[k] = gc[m_].mean(0)
        c = mean[segment_oracle(e, comp_xyz)]

    elif method.startswith("RePaint"):
        if method == "RePaint-vanilla":
            model, part = DDPM_VAN, np.zeros(len(xyz), np.int64)
        elif method == "RePaint-part":
            model, part = DDPM_PART, segment(SEG, xyz)
        else:                                                   # oracle seg
            model, part = DDPM_PART, segment_oracle(e, xyz)
        out = repaint_colours(model, xyz, part, known, col, seed=seed)
        c = out[n_vis:]
    else:
        raise ValueError(method)

    return np.concatenate([comp_xyz, np.asarray(c, np.float32)], -1)


if RUN_FULL_EXPERIMENT and COMP is not None:
    e0 = VAL[0]
    xyz0, known0, col0, nv0 = build_union(e0, COMP[0])
    _, diag = repaint_colours(DDPM_PART, xyz0, segment(SEG, xyz0), known0, col0,
                              seed=0, return_diag=True)
    print("RePaint diagnostics on the first validation object:")
    print(f"  union cloud      : {len(xyz0)} points ({nv0} visible + {len(COMP[0])} completed)")
    print(f"  part visibility  : "
          + ", ".join("nan" if v != v else f"{v:.2f}" for v in diag["part_visibility"]))
    print(f"  anchorless parts : {diag['anchorless'] or 'none'}"
          + ("   <- filled from the learned prior, not from a neighbouring part"
             if diag["anchorless"] else ""))
    print(f"  adaptive U       : {diag['U']} (base {JUMP_N_SAMPLE}) -> {diag['n_steps']} steps")

## §9 · Evaluation

Every method is scored through `spec_score` — the shared spec's single entry point — on the
**invented points only**, against the GT missing region. Rows carry the shared `EVAL_COLUMNS`
schema, so they concatenate with System A's with no reconciliation.

Only the RePaint methods are stochastic, so only they are repeated `EVAL_REPEATS` times; the
deterministic baselines are scored once and their row is reused, which saves time without changing
any number.

In [ ]:
def evaluate_all():
    rows, cache = [], {}
    det = {"NN-copy", "part-mean", "oracle ceiling"}
    for method in METHODS:
        reps = 1 if method in det else EVAL_REPEATS
        for rep in range(reps):
            t0 = time.time()
            for i, e in enumerate(VAL):
                pred = colour_completion(e, COMP[i], method, seed=SEED + 1000 * rep)
                if rep == 0 and i < 4:
                    cache[(method, i)] = pred
                gt_miss = missing_of(e)
                meta = dict(system=SYSTEM_NAME, method=method, rep=rep, obj=i, mid=e["mid"],
                            difficulty=e["difficulty"])
                rows.append(dict(**meta, region="missing",
                                 **spec_score(pred, gt_miss, seed=i,
                                              equalize_to=EQUALIZE_PRED_N)))
            print(f"  {method:28s} rep {rep+1}/{reps}  ({time.time()-t0:.1f}s)", flush=True)
        if reps == 1 and EVAL_REPEATS > 1:
            base = [r for r in rows if r["method"] == method]
            for rep in range(1, EVAL_REPEATS):
                rows.extend([{**r, "rep": rep} for r in base])
    df = pd.DataFrame(rows)
    return df[[c for c in EVAL_COLUMNS if c in df.columns]], cache


if RUN_FULL_EXPERIMENT and COMP is not None:
    t0 = time.time()
    EVAL, SAMPLES = evaluate_all()
    EVAL.to_csv(RESULTS_DIR / "tables/eval_raw.csv", index=False)
    print(f"\n{len(EVAL)} rows in {(time.time()-t0)/60:.1f} min "
          f"-> tables/eval_raw.csv (shared schema)")
    MAIN = EVAL.groupby("method")[list(METRIC_DIR)].mean().reindex(METHODS)
    print("\n--- MAIN TABLE · missing region ---\n")
    print(MAIN.round(4).to_string())
    MAIN.to_csv(RESULTS_DIR / "tables/main_table.csv")
else:
    EVAL = MAIN = SAMPLES = None
    print("skipped — set RUN_FULL_EXPERIMENT = True")

In [ ]:
# ---------------- paired comparisons between colouring methods ----------------
def paired(base, cond, df, n_boot=N_BOOT):
    out = []
    for met, direction in METRIC_DIR.items():
        a = df[df.method == base].groupby("obj")[met].mean()
        b = df[df.method == cond].groupby("obj")[met].mean()
        j = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
        if len(j) < 5:
            continue
        sign = -1 if direction == "lower" else 1
        denom = abs(j.a.mean()); rel = denom > 1e-12
        scale = (100.0 / denom) if rel else 1.0
        rng = np.random.default_rng(0)
        idx = rng.integers(0, len(j), (n_boot, len(j)))
        ba, bb = j.a.values[idx].mean(1), j.b.values[idx].mean(1)
        with np.errstate(divide="ignore", invalid="ignore"):
            boot = sign * (bb - ba) * (100.0 / np.abs(ba) if rel else 1.0)
        boot = boot[np.isfinite(boot)]
        if boot.size < 20:
            continue
        try:
            p = stats.wilcoxon(j.a, j.b).pvalue
        except Exception:
            p = np.nan
        lo, hi = np.percentile(boot, [2.5, 97.5])
        out.append(dict(metric=met, scale="%" if rel else "abs",
                        baseline=j.a.mean(), conditioned=j.b.mean(),
                        improvement=sign * (j.b.mean() - j.a.mean()) * scale,
                        ci_lo=lo, ci_hi=hi,
                        win_rate=float((sign * (j.b - j.a) > 0).mean()), wilcoxon_p=p,
                        verdict="helps" if lo > 0 else ("hurts" if hi < 0 else "unresolved")))
    return pd.DataFrame(out)


PIPE_COMPARISONS = [
    ("NN-copy", "RePaint-part", "P1", "Does the learned pipeline beat copying the nearest visible colour?"),
    ("RePaint-vanilla", "RePaint-part", "P2", "Does the PART PRIOR add anything? (Study 2's Question B2, in the real pipeline)"),
    ("RePaint-part", "RePaint-part (oracle seg)", "P3", "How much is lost to SEGMENTATION error?"),
    ("part-mean", "RePaint-part", "P4", "Does the learned model beat the hand-written part-mean rule?"),
    ("RePaint-part (oracle seg)", "oracle ceiling", "P5", "Distance to the best any part-constant rule can do"),
]

if EVAL is not None:
    STATS = []
    for base, cond, tag, q in PIPE_COMPARISONS:
        if base not in METHODS or cond not in METHODS:
            continue
        s = paired(base, cond, EVAL).assign(comparison=tag, baseline_method=base, cond_method=cond)
        STATS.append(s)
        print("=" * 96); print(f"{tag}: {base}  ->  {cond}"); print(q); print("=" * 96)
        print(s[["metric", "baseline", "conditioned", "improvement", "ci_lo", "ci_hi",
                 "win_rate", "wilcoxon_p", "verdict"]].round(4).to_string(index=False)); print()
    STATS = pd.concat(STATS, ignore_index=True)
    STATS.to_csv(RESULTS_DIR / "tables/paired_stats.csv", index=False)
    print("A CI spanning 0 is NOT resolved at this sample size; ci_hi < 0 is resolved as HARMFUL.")

## §10 · Visualisations

In [ ]:
def _view(ax, xyz, rgb, title="", s=4, lim=None):
    o = np.argsort(xyz[:, 1])
    ax.scatter(xyz[o, 0], xyz[o, 2], c=np.clip(rgb[o], 0, 1), s=s, linewidths=0)
    ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, fontsize=8)
    if lim: ax.set_xlim(lim[0]); ax.set_ylim(lim[1])


if SAMPLES:
    n_show = min(3, len(VAL))
    ncol = 2 + len(METHODS)
    fig, axes = plt.subplots(n_show, ncol, figsize=(2.6 * ncol, 2.7 * n_show))
    axes = np.atleast_2d(axes)
    for r in range(n_show):
        e = VAL[r]; g = e["x0"]; vis = e["vis"]
        lim = ([g[:, 0].min() - .1, g[:, 0].max() + .1], [g[:, 2].min() - .1, g[:, 2].max() + .1])
        _view(axes[r, 0], g[vis, :3], (g[vis, 3:] + 1) / 2,
              f"input ({vis.mean():.0%} seen)" if r == 0 else "", lim=lim)
        for c, meth in enumerate(METHODS):
            p = SAMPLES.get((meth, r))
            if p is not None:
                _view(axes[r, c + 1], p[:, :3], (p[:, 3:] + 1) / 2,
                      meth.replace(" (oracle seg)", "\n(oracle seg)") if r == 0 else "", lim=lim)
            else:
                axes[r, c + 1].axis("off")
        _view(axes[r, -1], g[:, :3], (g[:, 3:] + 1) / 2, "ground truth" if r == 0 else "", lim=lim)
    fig.suptitle(f"Colouring methods on IDENTICAL geometry — {CATEGORY}, {DIFFICULTIES[0]} "
                 f"(geometry: {GEOMETRY_SOURCE})", fontsize=11)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/10_methods.png", dpi=140,
                                    bbox_inches="tight"); plt.close(fig)
    print("saved figures/10_methods.png")

In [ ]:
if EVAL is not None:
    show = ["cd_l1", "fscore", "dE00_sym", "colour_swd", "region_dE", "colour_edge_iou"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for ax, met in zip(axes.ravel(), show):
        vals, errs = [], []
        for m in METHODS:
            per = EVAL[EVAL.method == m].groupby("obj")[met].mean().dropna().values
            if len(per) == 0:
                vals.append(np.nan); errs.append([0, 0]); continue
            rng = np.random.default_rng(0)
            bs = per[rng.integers(0, len(per), (2000, len(per)))].mean(1)
            vals.append(per.mean())
            errs.append([per.mean() - np.percentile(bs, 2.5), np.percentile(bs, 97.5) - per.mean()])
        ax.bar(range(len(METHODS)), vals, yerr=np.array(errs).T, capsize=3,
               color=["#b8b8b8", "#9ab8d8", "#6699cc", "#2f6f4f", "#4a9c74", "#d9a441"])
        ax.set_xticks(range(len(METHODS)))
        ax.set_xticklabels([m.replace(" (oracle seg)", "\n(oracle)") for m in METHODS],
                           fontsize=7, rotation=25, ha="right")
        ax.set_title(f"{met}  ({'lower' if METRIC_DIR[met]=='lower' else 'higher'} better)", fontsize=9)
        ax.grid(alpha=.25, axis="y")
    fig.suptitle("Missing region · mean over objects, 95% bootstrap CI", fontsize=11)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/10_headline.png", dpi=130,
                                    bbox_inches="tight"); plt.close(fig)
    print("saved figures/10_headline.png")

    # colour error vs segmentation accuracy — does label quality drive the part methods?
    if not np.isnan(SEG_ACC):
        acc = pd.read_csv(RESULTS_DIR / "tables/segmentation_accuracy.csv")
        fig, ax = plt.subplots(figsize=(6, 3.6))
        for m in ["RePaint-part", "RePaint-part (oracle seg)", "part-mean"]:
            if m not in METHODS: continue
            d = EVAL[EVAL.method == m].groupby("obj").dE00_sym.mean()
            ax.scatter(acc.acc.values[:len(d)], d.values, s=18, alpha=.7, label=m)
        ax.set_xlabel("per-object part-seg accuracy"); ax.set_ylabel("dE00_sym (lower better)")
        ax.legend(fontsize=7); ax.grid(alpha=.3)
        ax.set_title("Is segmentation quality the binding constraint?", fontsize=10)
        fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/10_seg_vs_error.png", dpi=130)
        plt.close(fig); print("saved figures/10_seg_vs_error.png")

## §11 · Report

In [ ]:
def md_table(df, index=False):
    try:
        return df.to_markdown(index=index)
    except Exception:
        d = df.reset_index() if index else df
        cols = [str(c) for c in d.columns]
        f = lambda v: ("nan" if v != v else f"{v:.4g}") if isinstance(v, float) else str(v)
        return "\n".join(["| " + " | ".join(cols) + " |",
                          "|" + "|".join("---" for _ in cols) + "|"]
                         + ["| " + " | ".join(f(v) for v in r) + " |"
                            for r in d.itertuples(index=False)])


def build_report():
    L = [f"# RePaint-part Pipeline (System B) — {CATEGORY}", "",
         f"*Shared spec `{SPEC_HASH}` · config `{CFG_HASH}`. Both must match System A's.*", "",
         "## 1. Setup", "",
         f"- geometry stage: **{GEOMETRY_SOURCE}**"
         + ("  (frozen PoinTr, ShapeNet-55 checkpoint)" if GEOMETRY_SOURCE == "pointr"
            else "  (**perfect geometry** — colour comparison only)"),
         f"- {len(TRAIN)} train / {len(VAL)} val, {NUM_POINTS} points, seed {SEED}, "
         f"difficulties {DIFFICULTIES}",
         f"- part-seg held-out accuracy: **{SEG_ACC:.1%}**" if SEG_ACC == SEG_ACC else
         "- part-seg not trained", ""]
    if GEOMETRY_SOURCE == "pointr" and RANK:
        L += [f"- axis frame chosen by measurement: perm {AXIS_PERM}, sign {AXIS_SIGN} "
              f"(best of 48 by Chamfer; margin over runner-up "
              f"{RANK[1][0]/RANK[0][0]-1:.1%})", ""]
    if EVAL is None:
        L += ["## Results", "", "Not run. Set `RUN_FULL_EXPERIMENT = True` in §1."]
        return "\n".join(L)

    L += ["## 2. Main table — missing region", "", md_table(MAIN.round(4), index=True), "",
          "All six methods share identical geometry, so every difference is a difference in "
          "**colouring**.", "", "## 3. Paired comparisons", ""]
    for base, cond, tag, q in PIPE_COMPARISONS:
        s = STATS[STATS.comparison == tag]
        if not len(s): continue
        L += [f"### {tag} · `{base}` → `{cond}`", "", f"*{q}*", "",
              md_table(s[["metric", "baseline", "conditioned", "improvement", "ci_lo", "ci_hi",
                          "win_rate", "wilcoxon_p", "verdict"]].round(4)), ""]

    p2 = STATS[STATS.comparison == "P2"]
    part_verdict = "unresolved"
    if len(p2):
        h, x = (p2.verdict == "helps").sum(), (p2.verdict == "hurts").sum()
        part_verdict = ("helps" if h > x and h >= 2 else "hurts" if x > h and x >= 2
                        else "unresolved on every metric" if h == x == 0 else "mixed")
    L += ["## 4. What this run says about the part prior", "",
          f"`RePaint-vanilla` → `RePaint-part` (P2) came back **{part_verdict}**.", "",
          "The controlled study predicted this: adding part identity as *context* was Question B2, "
          "which was neutral on all 8 metrics. Only *restricting* aggregation to same-part "
          "neighbours was harmful (Question C, win rate 2.6%), and this pipeline's `part_pool` is "
          "additive, not restrictive — so it sits on the neutral side of that line.", "",
          f"P3 isolates how much is lost to segmentation error at {SEG_ACC:.1%} accuracy, and P5 "
          "gives the distance to the best any part-constant rule could achieve.", "",
          "## 5. Limits", "",
          "1. **PoinTr's checkpoint was trained on ShapeNet-55**, which contains these categories "
          "and plausibly these exact models. The geometry stage may have memorised part of the "
          "evaluation set. Its geometry numbers are an optimistic bound and must not be read as a "
          "fair comparison against a from-scratch system.",
          "2. **The colour DDPM never sees occlusion during training** — it meets the mask only at "
          "inference, through RePaint.",
          "3. One seed, one category, "
          f"{len(VAL)} validation objects, {len(METRIC_DIR)}×{len(PIPE_COMPARISONS)} uncorrected tests.",
          "4. Tier-2 colour metrics move with geometry error; read them beside `match_rate` and "
          "against the correspondence-free tier-3 numbers.", ""]
    return "\n".join(L)


rep = build_report()
(RESULTS_DIR / "report.md").write_text(rep)
print(rep[:3000])
print(f"\n... full report at {RESULTS_DIR/'report.md'} ({len(rep)} chars)")

In [ ]:
print("=" * 78); print("SAVED OUTPUTS"); print("=" * 78)
for sub in ["", "tables", "figures", "checkpoints"]:
    d = RESULTS_DIR / sub
    fs = sorted(f for f in d.iterdir() if f.is_file()) if d.is_dir() else []
    if fs:
        print(f"\n{sub or 'report'}  ({d})")
        for f in fs:
            print(f"  {f.name:<44} {f.stat().st_size/1024:>9.1f} KB")
print(f"""
==============================================================================
FOR THE COMPARISON
==============================================================================
Hand these to Comparison_Report_kaggle.ipynb:
  {RESULTS_DIR}/tables/eval_raw.csv
  {RESULTS_DIR}/run_manifest.json      (spec {SPEC_HASH} / config {CFG_HASH})

The comparison notebook checks BOTH hashes against System A's and refuses to merge
if either differs — that check is the whole reason these numbers can be compared.""")